# EDA — FIFA World Rankings & FiveThirtyEight SPI
**Sources**:
- `fifa_rankings/` — FIFA official ranking snapshots (2023–2024 CSVs)
- `fivethirtyeight_spi/` — FiveThirtyEight Soccer Power Index: `spi_global_rankings_intl.csv`, `spi_matches.csv`, `spi_matches_latest.csv`

**Purpose**: Compare FIFA rankings and SPI as alternative team strength features to ELO. Understand coverage, update frequency, and correlation with ELO.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

sns.set_theme(style='whitegrid', palette='muted')
FIFA_DATA = Path('../data/raw/fifa_rankings')
SPI_DATA = Path('../data/raw/fivethirtyeight_spi')

## 1. FIFA Rankings — Load All Snapshots

In [ ]:
fifa_files = sorted([f for f in os.listdir(FIFA_DATA) if f.endswith('.csv')])
print('FIFA ranking files:', fifa_files)

fifa_dfs = []
for f in fifa_files:
    df = pd.read_csv(FIFA_DATA / f, parse_dates=['rank_date'])
    fifa_dfs.append(df)

fifa = pd.concat(fifa_dfs, ignore_index=True)
print('Combined shape:', fifa.shape)
print('Columns:', fifa.columns.tolist())
print('Date range:', fifa['rank_date'].min(), '->', fifa['rank_date'].max())
print('Unique teams:', fifa['country_full'].nunique())
fifa.head(3)

In [ ]:
print('Missing values:')
print(fifa.isnull().mean().mul(100).round(2))
print()
print('Points range:', fifa['total_points'].min(), '-', fifa['total_points'].max())
print('Rank range:', fifa['rank'].min(), '-', fifa['rank'].max())

## 2. FIFA Rankings — Latest Snapshot Top 30

In [ ]:
latest_snapshot_date = fifa['rank_date'].max()
latest_fifa = fifa[fifa['rank_date'] == latest_snapshot_date].sort_values('rank')

print(f'Latest snapshot: {latest_snapshot_date.date()}')
print('\nTop 30:')
print(latest_fifa[['rank', 'country_full', 'total_points', 'confederation']].head(30).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
latest_fifa.head(30).sort_values('total_points').plot(
    kind='barh', x='country_full', y='total_points', ax=ax, legend=False, color='steelblue'
)
ax.set_title(f'Top 30 FIFA Rankings by Points ({latest_snapshot_date.date()})')
ax.set_xlabel('FIFA Points')
plt.tight_layout()
plt.show()

## 3. FIFA Rankings — Point Distribution & Confederation Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

latest_fifa['total_points'].plot(kind='hist', bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('FIFA Points Distribution')
axes[0].set_xlabel('Points')

conf_means = latest_fifa.groupby('confederation')['total_points'].mean().sort_values()
conf_means.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Mean FIFA Points by Confederation')
axes[1].set_xlabel('Mean Points')

plt.tight_layout()
plt.show()

## 4. Snapshot Stability — Rank Changes Between Snapshots

In [ ]:
pivot = fifa.pivot_table(index='country_full', columns='rank_date', values='rank')
print('Pivot shape (teams x snapshots):', pivot.shape)

if pivot.shape[1] >= 2:
    cols = sorted(pivot.columns)
    rank_diff = pivot[cols[-1]] - pivot[cols[0]]
    rank_diff = rank_diff.dropna()
    
    print(f'\nRank change {cols[0].date()} -> {cols[-1].date()}:')
    print(f'Mean abs change: {rank_diff.abs().mean():.1f}')
    print(f'Max rise: {rank_diff.min():.0f} ({rank_diff.idxmin()})')
    print(f'Max fall: {rank_diff.max():.0f} ({rank_diff.idxmax()})')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    rank_diff.plot(kind='hist', bins=40, ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title(f'FIFA Rank Change Distribution ({cols[0].date()} to {cols[-1].date()})')
    ax.set_xlabel('Rank Change (negative = improved)')
    plt.tight_layout()
    plt.show()
else:
    print('Only one snapshot available — cannot compute stability')

## 5. FiveThirtyEight SPI — Load & Profile

In [ ]:
spi_files = [f for f in os.listdir(SPI_DATA) if f.endswith('.csv')]
print('SPI files:', spi_files)

spi_rankings = pd.read_csv(SPI_DATA / 'spi_global_rankings_intl.csv')
spi_matches = pd.read_csv(SPI_DATA / 'spi_matches.csv', parse_dates=['date'])

# Filter matches to international only (no league column means intl)
print('SPI rankings shape:', spi_rankings.shape)
print('SPI rankings columns:', spi_rankings.columns.tolist())
print()
print('SPI matches shape:', spi_matches.shape)
print('SPI matches date range:', spi_matches['date'].min(), '->', spi_matches['date'].max())
spi_rankings.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

spi_rankings['spi'].plot(kind='hist', bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('SPI Distribution')
axes[0].set_xlabel('Soccer Power Index')

spi_rankings['off'].plot(kind='hist', bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Offensive Rating Distribution')
axes[1].set_xlabel('Offensive Rating')

spi_rankings['def'].plot(kind='hist', bins=40, ax=axes[2], color='green', edgecolor='white', alpha=0.7)
axes[2].set_title('Defensive Rating Distribution')
axes[2].set_xlabel('Defensive Rating (lower = better)')

plt.tight_layout()
plt.show()

print('SPI Top 20:')
print(spi_rankings.sort_values('spi', ascending=False)[['name', 'confed', 'spi', 'off', 'def']].head(20).to_string(index=False))

## 6. SPI Projected Scores — Calibration Check

In [ ]:
spi_with_results = spi_matches.dropna(subset=['score1', 'score2', 'proj_score1', 'proj_score2'])
print(f'Matches with both projected and actual scores: {len(spi_with_results)}')

if len(spi_with_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, proj_col, actual_col, label in [
        (axes[0], 'proj_score1', 'score1', 'Team 1 Goals'),
        (axes[1], 'proj_score2', 'score2', 'Team 2 Goals'),
    ]:
        ax.scatter(spi_with_results[proj_col], spi_with_results[actual_col], alpha=0.2, s=5)
        ax.set_xlabel(f'Projected {label}')
        ax.set_ylabel(f'Actual {label}')
        ax.set_title(f'SPI Projected vs Actual: {label}')
        # correlation
        r = spi_with_results[proj_col].corr(spi_with_results[actual_col])
        ax.text(0.05, 0.95, f'r={r:.3f}', transform=ax.transAxes, va='top')

    plt.tight_layout()
    plt.show()

## 7. FIFA Rankings vs SPI — Coverage Overlap

In [ ]:
fifa_teams = set(latest_fifa['country_full'].str.strip())
spi_teams = set(spi_rankings['name'].str.strip())

print('FIFA teams:', len(fifa_teams))
print('SPI teams:', len(spi_teams))
print('In both:', len(fifa_teams & spi_teams))
print('Only in FIFA:', len(fifa_teams - spi_teams))
print('Only in SPI:', len(spi_teams - fifa_teams))
print()
print('Note: Name mismatches expected — need harmonization before joining')

## 8. Red Flags Summary

In [ ]:
import datetime
red_flags = []
today = datetime.date(2026, 6, 12)

days_stale = (today - latest_snapshot_date.date()).days
if days_stale > 60:
    red_flags.append(f'FIFA rankings data is {days_stale} days stale (last snapshot: {latest_snapshot_date.date()}) — missing 2025–2026 qualifier period; use for historical training only')

if spi_matches['date'].max().year < 2024:
    red_flags.append(f'SPI matches data ends {spi_matches["date"].max().date()} — FiveThirtyEight shut down SPI in 2023; use for validation/calibration reference only, not current team strength')

if red_flags:
    print('RED FLAGS:')
    for f in red_flags:
        print(' -', f)
else:
    print('No red flags.')

print()
print('MODELING NOTE: FIFA rankings are useful as a secondary feature but are less predictive than ELO.')
print('SPI (discontinued 2023) is best used as a calibration reference for historical validation.')